In [1]:
import sys
import os
import gc
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

from preprocessing.Zhou2016.preprocessing import (
    load_zhou2016_data
)
from preprocessing.general.filtering import (
    apply_filters_to_dataset,
    bands
)
import preprocessing.general.feature_extraction as fe


# ============================================================
# Dataset metadata
# ============================================================

DATASET_NAME = "zhou2016"

SESSIONS = [
    "session_01",
    "session_02",
    "session_03",
]


# ============================================================
# Channel configuration
# ============================================================

ELECTRODE_SETUP = "setup_01"

selected_channels = [
    "C3",
    "Cz",
    "C4",
]

# Use all available Zhou2016 EEG channels instead:
# selected_channels = None


# ============================================================
# Paths
# ============================================================

root_zhou = os.path.join(
    PROJECT_ROOT,
    "Datasets/Zhou2016/original/Zhou2016/"
)

output_csv = Path(
    os.path.join(
        PROJECT_ROOT,
        "Datasets/Zhou2016/processed/",
        f"Zhou2016_features_{ELECTRODE_SETUP}.csv",
    )
)


# ============================================================
# Feature configuration
# ============================================================

extract_config = {
    "mean": {"function": fe.extract_mean},
    "std": {"function": fe.extract_std},
    "mom": {"function": fe.extract_moments},
    "min": {"function": fe.extract_min},
    "max": {"function": fe.extract_max},
    "cov": {"function": fe.extract_covariance},
    "eig": {"function": fe.extract_eigenvalues},

    # "logcov": {
    #     "function": fe.extract_logcov
    # },

    # "fft": {
    #     "function": fe.extract_fft,
    #     "params": {"ntop": 5},
    # },

    "h_diff": {"function": fe.extract_halves_diff},
    "q_stats": {"function": fe.extract_quarters_stats},
    "logvar": {"function": fe.extract_logvar},
}


# ============================================================
# Incremental processing configuration
# ============================================================

subjects = list(range(1, 5))

# Only four subjects, so this can remain small.
SUBJECT_BATCH_SIZE = 2


# ============================================================
# Prepare output
# ============================================================

output_csv.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if output_csv.exists():
    output_csv.unlink()

first_write = True
total_rows = 0


# ============================================================
# Load, filter, extract, validate, and save incrementally
# ============================================================

for start in range(
    0,
    len(subjects),
    SUBJECT_BATCH_SIZE,
):

    subject_batch = subjects[
        start:start + SUBJECT_BATCH_SIZE
    ]

    print(
        f"\nProcessing subjects "
        f"{subject_batch[0]}–{subject_batch[-1]}"
    )

    # --------------------------------------------------------
    # Load current batch
    # --------------------------------------------------------

    batch_data = load_zhou2016_data(
        root=root_zhou,
        config={
            "subjects": subject_batch,
            "channels": selected_channels,
        },
    )

    if not batch_data:
        print("⚠️ No data loaded for this batch.")
        continue

    print("✅ Data loading complete.")

    # --------------------------------------------------------
    # Filtering and resampling
    # --------------------------------------------------------

    filtered_data = apply_filters_to_dataset(
        dataset=batch_data,
        config={
            "original_fs": 250,
        },
    )

    print("✅ Filtering complete.")

    # --------------------------------------------------------
    # Feature extraction per session
    # --------------------------------------------------------

    session_frames = []

    for session_name in SESSIONS:

        session_data = {}

        for subject_id, subject_sessions in filtered_data.items():

            if session_name not in subject_sessions:
                continue

            session_data[subject_id] = {
                session_name: subject_sessions[session_name]
            }

        if not session_data:
            continue

        df_session = fe.extract_features_to_dataframe(
            dataset=session_data,
            extract_config=extract_config,
            band_labels=bands,
            dataset_name=DATASET_NAME,
            session_name=session_name,
        )

        if not df_session.empty:
            session_frames.append(df_session)

    if not session_frames:
        print("⚠️ No features generated for this batch.")

        del batch_data
        del filtered_data

        gc.collect()
        continue

    df_batch = pd.concat(
        session_frames,
        ignore_index=True,
    )

    print(
        f"✅ Feature extraction complete: "
        f"{df_batch.shape}"
    )

    # --------------------------------------------------------
    # Validate current batch
    # --------------------------------------------------------

    fe.validate_feature_dataframe(df_batch)

    print("✅ Batch validation complete.")

    # --------------------------------------------------------
    # Append batch to CSV
    # --------------------------------------------------------

    df_batch.to_csv(
        output_csv,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    if first_write:
        display(df_batch.head())
        first_write = False

    total_rows += len(df_batch)

    print(
        f"✅ Batch saved. "
        f"Total rows written: {total_rows}"
    )

    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del batch_data
    del filtered_data
    del session_frames
    del df_batch

    gc.collect()


# ============================================================
# Final validation and report
# ============================================================

if first_write:
    print("⚠️ No feature data were written.")

else:
    print("\nValidating complete saved dataset...")

    df_complete = pd.read_csv(output_csv)

    fe.validate_feature_dataframe(df_complete)

    if len(df_complete) != total_rows:
        raise ValueError(
            "The number of rows in the saved CSV does not match "
            f"the number written: {len(df_complete)} versus "
            f"{total_rows}."
        )

    print("✅ Complete dataset validation passed.")
    print("✅ Complete feature extraction finished.")
    print(f"✅ Electrode setup: {ELECTRODE_SETUP}")
    print(f"✅ Channels: {selected_channels}")
    print(f"✅ Total rows: {total_rows}")
    print(f"✅ Saved to: {output_csv}")

    print("\nRows per session:")
    print(
        df_complete["session"]
        .value_counts()
        .sort_index()
    )

    del df_complete
    gc.collect()


Processing subjects 1–2
✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 2/2 [00:00<00:00,  4.56it/s]

✅ Feature extraction complete: (914, 95)
✅ Batch validation complete.


,dataset,subject,session,trial_index,label,b8_12_mean_C3,b8_12_mean_Cz,b8_12_mean_C4,b8_12_std_C3,b8_12_std_Cz,...,b13_30_q_1_C4,b13_30_q_2_C3,b13_30_q_2_Cz,b13_30_q_2_C4,b13_30_q_3_C3,b13_30_q_3_Cz,b13_30_q_3_C4,b13_30_logvar_C3,b13_30_logvar_Cz,b13_30_logvar_C4
0,zhou2016,subject_01,session_01,0,both_feet_imagery,1.214972e-08,9.017326e-09,-1.942930e-08,0.000002,0.000002,...,-9.927847e-08,-3.992778e-08,2.701131e-08,9.721808e-08,2.412391e-08,8.294311e-08,6.550031e-08,-24.550291,-24.556559,-24.592585
1,zhou2016,subject_01,session_01,1,left_hand_imagery,2.953547e-09,-2.016497e-09,-7.963990e-09,0.000002,0.000002,...,1.760957e-09,-6.489544e-08,-6.656614e-08,-2.336503e-08,-4.436873e-08,-2.707120e-08,-6.005128e-08,-25.030989,-25.017496,-24.970583
2,zhou2016,subject_01,session_01,2,left_hand_imagery,7.881542e-09,6.649660e-09,2.691711e-09,0.000002,0.000002,...,-6.711952e-08,5.268241e-08,6.226033e-08,4.763525e-08,3.519465e-09,-3.688134e-08,-2.669045e-09,-24.755386,-24.548975,-24.309647
3,zhou2016,subject_01,session_01,3,right_hand_imagery,-8.720858e-09,-2.809748e-09,3.942637e-09,0.000002,0.000002,...,2.291055e-08,-8.242959e-08,-2.233984e-09,2.234947e-08,1.215385e-07,8.500663e-08,1.355512e-08,-24.869326,-24.940140,-24.724112
4,zhou2016,subject_01,session_01,4,both_feet_imagery,1.271896e-08,1.182335e-08,1.212933e-08,0.000002,0.000002,...,1.220686e-08,-1.004149e-07,-8.823270e-08,-7.538063e-08,6.774042e-08,7.552958e-08,1.339844e-07,-25.010378,-25.048298,-25.044220


✅ Batch saved. Total rows written: 914

Processing subjects 3–4
✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 2/2 [00:00<00:00,  2.75it/s]


✅ Feature extraction complete: (886, 95)
✅ Batch validation complete.
✅ Batch saved. Total rows written: 1800

Validating complete saved dataset...
✅ Complete dataset validation passed.
✅ Complete feature extraction finished.
✅ Electrode setup: setup_01
✅ Channels: ['C3', 'Cz', 'C4']
✅ Total rows: 1800
✅ Saved to: ../../../Datasets/Zhou2016/processed/Zhou2016_features_setup_01.csv

Rows per session:
session
session_01    614
session_02    586
session_03    600
Name: count, dtype: int64
